## Lineare Regression

### Aufgabenstellung

Eine Photovoltaikanlage liefert alle 15 Minuten zwei Messwerte: die Sonneneinstrahlung (`irradiation`) aus einem Wettersensor und die Leistung der ganzen Anlage (`ac_power`, Summe aller Wechselrichter). In dieser Aufgabe schätzt du die Leistung der Anlage aus der Einstrahlung. Der Datensatz enthält nur Messungen bei Tageslicht.

Hier findest du die relevanten Dokumentationen dazu:
- [Korrelation](https://abbts-dat-ski.github.io/wissensdatenbank/statistics/correlation/)
- [Grundlagen Machine Learning](https://abbts-dat-ski.github.io/wissensdatenbank/machine-learning/basics/)
- [Lineare Regression](https://abbts-dat-ski.github.io/wissensdatenbank/machine-learning/linear_regression/)
- [Residuenanalyse](https://abbts-dat-ski.github.io/wissensdatenbank/machine-learning/residual_analysis/)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv("../../data/ub8_pv_irradiation.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df[["timestamp", "irradiation", "ac_power"]].head()

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["irradiation"],
        y=df["ac_power"],
        mode="markers",
        name="Messpunkte",
        marker={"opacity": 0.45},
    )
)

fig.update_layout(
    title="Einstrahlung und Leistung der PV-Anlage",
    xaxis_title="irradiation",
    yaxis_title="ac_power",
)

fig.show()

### Aufgabe 1
Prüfe zuerst mit einer Korrelation, ob ein linearer Zusammenhang zwischen `irradiation` und `ac_power` plausibel ist. Schätze danach `ac_power` mit `irradiation`: Erstelle eine Regressionsgerade, gib Achsenabschnitt und Steigung aus und erkläre, was die Steigung für die Anlage bedeutet.

In [ ]:
# Dein Code hier

In [ ]:
df[["irradiation", "ac_power"]].corr()

In [ ]:
model = LinearRegression()
model.fit(df[["irradiation"]], df["ac_power"])

df["prediction"] = model.predict(df[["irradiation"]])
plot_df = df.sort_values("irradiation")

print(f"Achsenabschnitt b: {model.intercept_}")
print(f"Steigung m: {model.coef_[0]}")

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["irradiation"],
        y=df["ac_power"],
        mode="markers",
        name="Messpunkte",
        marker={"opacity": 0.45},
    )
)
fig.add_trace(
    go.Scatter(
        x=plot_df["irradiation"],
        y=plot_df["prediction"],
        mode="lines",
        name="Regressionsgerade",
        line={"color": "crimson", "width": 3},
    )
)

fig.update_layout(
    title="Einstrahlung und Leistung mit Regressionsgerade",
    xaxis_title="irradiation",
    yaxis_title="ac_power",
)

fig.show()

Die Pearson-Korrelation liegt bei etwa 0.99, die Punkte liegen fast auf einer Geraden. Die Steigung gibt an, um wie viel die Leistung der Anlage steigt, wenn die Einstrahlung um eine Einheit zunimmt. Der Achsenabschnitt ist im Vergleich zur Maximalleistung von rund 29 000 klein: ohne Einstrahlung liefert die Anlage praktisch keine Leistung.

### Aufgabe 2
Untersuche mit einer Residuen-Grafik, wo das Modell gut passt und wo nicht. Beschreibe kurz, was dir auffällt.

In [ ]:
# Dein Code hier

In [ ]:
df["residuals"] = df["ac_power"] - df["prediction"]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["irradiation"],
        y=df["residuals"],
        mode="markers",
        name="Residuen",
        marker={"opacity": 0.45},
    )
)
fig.add_hline(y=0, line_color="crimson", line_dash="dash")

fig.update_layout(
    title="Residuen des Modells",
    xaxis_title="irradiation",
    yaxis_title="Residuen",
)

fig.show()

Die Residuen liegen grösstenteils nahe bei 0. Einzelne Ausreisser entstehen zum Beispiel, wenn Wolken die Einstrahlung schnell verändern und der Wettersensor und die Wechselrichter nicht genau dasselbe 15-Minuten-Mittel messen.

### Aufgabe 3
Fasse den Modellfehler mit MAE und MSE zusammen. Vergleiche den MAE mit der typischen Leistung der Anlage und erkläre kurz, weshalb MSE stärker auf grosse Abweichungen reagiert.

In [ ]:
# Dein Code hier

In [ ]:
mae = mean_absolute_error(df["ac_power"], df["prediction"])
mse = mean_squared_error(df["ac_power"], df["prediction"])

print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Mittlere Leistung: {df['ac_power'].mean()}")

Der MAE ist im Vergleich zur mittleren Leistung klein, das Modell ist für eine Schätzung gut brauchbar. Der MSE gewichtet grosse Fehler stärker, weil die Abweichungen quadriert werden. Einzelne Wolkenmomente fallen darum im MSE stärker ins Gewicht als im MAE.
